# Polymarket BTC Inefficiency Backtest (Colab)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Optional: clone your repo into Colab runtime
GIT_REPO_URL = ''  # e.g. 'https://github.com/<org>/<repo>.git'
REPO_DIR = '/content/BTC-Polymarket'

import os, sys
from pathlib import Path
if GIT_REPO_URL:
    !git clone $GIT_REPO_URL $REPO_DIR

os.chdir(REPO_DIR)
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
print('Using repo at', REPO_DIR)


In [ ]:
# Config
START_DATE = '2024-01-01'
END_DATE = '2024-04-30'
ENTRY_THRESHOLD = 0.05
EXIT_THRESHOLD = 0.01
OUT_PATH = '/content/drive/MyDrive/polymarket_btc_inefficiency/data/pm_btc_reference.parquet'
REPORT_PATH = '/content/drive/MyDrive/polymarket_btc_inefficiency/backtest_report.md'


In [ ]:
from polymarket_btc.dataset import build_dataset
from polymarket_btc.backtest import BacktestConfig, simulate_backtest, write_report

rows = build_dataset(OUT_PATH, start=START_DATE, end=END_DATE)
cfg = BacktestConfig(entry_threshold=ENTRY_THRESHOLD, exit_threshold=EXIT_THRESHOLD)
bt, metrics = simulate_backtest(rows, cfg)
write_report(metrics, REPORT_PATH)
print('Rows:', len(rows))
print('Dataset:', OUT_PATH)
print('Report:', REPORT_PATH)
print(metrics)


In [ ]:
# Minimal plotting without external dependencies
# (uses matplotlib if available in Colab runtime)
try:
    import matplotlib.pyplot as plt
    eq=[]
    x=[]
    cur=1.0
    for r in bt:
        cur *= (1+r['pnl'])
        eq.append(cur)
        x.append(r['ts'])
    plt.figure(figsize=(10,4))
    plt.plot(x, eq)
    plt.title('Equity Curve')
    plt.grid(True)
    plt.show()
except Exception as exc:
    print('Plot skipped:', exc)


In [ ]:
verdict = 'yes' if metrics['sharpe'] > 0.5 and metrics.get('oos_sharpe', 0) > 0 else 'unclear'
survives_costs = 'yes' if metrics.get('avg_trade_return', 0) > 0 else 'no/unclear'
print('=== Final Verdict ===')
print('evidence for inefficiency:', verdict)
print('whether edge survives costs:', survives_costs)
